# 05-2 Llama 모델로 텍스트 생성하기

<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/rickiepark/hm-dl/blob/main/05-2-llama2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

코랩에서 이 노트북을 실행하려면 High-RAM CPU 런타임을 사용해야 합니다.

In [1]:
import keras
from keras import layers
import keras_nlp

In [2]:
def make_causal_mask(seq_len):
    n_hori = keras.ops.arange(seq_len)
    n_vert = keras.ops.expand_dims(n_hori, axis=-1)
    mask = n_vert >= n_hori
    return mask

In [3]:
def make_attention_mask(padding_mask):
    # padding_mask 크기가 (2, 5)라고 가정해 보죠.
    batch_size, seq_len = keras.ops.shape(padding_mask)
    # causal_mask 크기는 (5, 5)가 됩니다.
    causal_mask = make_causal_mask(seq_len)
    # 배치 차원을 추가해 (2, 5, 5)로 만듭니다.
    causal_mask = keras.ops.broadcast_to(causal_mask, (batch_size, seq_len, seq_len))
    # 브로드캐스팅을 위해 padding_mask 크기를 (2, 1, 5)로 만듭니다.
    padding_mask = keras.ops.expand_dims(padding_mask, axis=1)
    return keras.ops.minimum(causal_mask, padding_mask)

In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class AttentionMask(layers.Layer):
    def call(self, padding_mask):
        return make_attention_mask(padding_mask)

# class AttentionMask(keras.Layer):
#     def call(self, padding_mask):
#         return make_attention_mask(padding_mask)

## Llama 모델 이해하기

### 로터리 위치 임베딩

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 토큰 임베딩 크기
embed_dim = 4096

class RotaryEmbedding(layers.Layer):

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]

        inv_freq = 1.0 / (
            10000 ** (tf.range(0, self.dim, 2, dtype=tf.float32) / self.dim)
        )

        pos = tf.range(seq_len, dtype=tf.float32)

        freqs = tf.einsum("i,j->ij", pos, inv_freq)

        cos = tf.cos(freqs)
        sin = tf.sin(freqs)

        cos = tf.reshape(cos, (1, seq_len, self.dim // 2))
        sin = tf.reshape(sin, (1, seq_len, self.dim // 2))

        x1, x2 = tf.split(inputs, 2, axis=-1)

        out1 = x1 * cos - x2 * sin
        out2 = x1 * sin + x2 * cos

        return tf.concat([out1, out2], axis=-1)

# 가상의 토큰 임베딩
inputs = tf.ones((1, 2, embed_dim), dtype=tf.float32)

# 두 번째 위치에 있는 토큰에 로터리 위치 임베딩 적용
rotary_embedding = RotaryEmbedding(embed_dim)
output = rotary_embedding(inputs)
print(output.shape)
print(output[:10])

# # 토큰 임베딩 크기
# embed_dim = 4096

# def rotary_position_embedding(inputs, token_pos):
#     # theta 각도를 생성합니다.
#     freqs = keras.ops.arange(0, embed_dim, 2, dtype='float32') / embed_dim
#     inverse_freqs = 1 / (10000**freqs)
#     # m * theta
#     embedding = token_pos * inverse_freqs
#     cos_emb = keras.ops.cos(embedding)
#     sin_emb = keras.ops.sin(embedding)
#     # 입력을 절반으로 나눕니다.
#     x1, x2 = keras.ops.split(inputs, 2)
#     # 회전 변환을 적용합니다.
#     new_x1 = x1 * cos_emb - x2 * sin_emb
#     new_x2 = x1 * sin_emb + x2 * cos_emb
#     return keras.ops.concatenate((new_x1, new_x2))

# # 가상의 토큰 임베딩
# inputs = keras.ops.ones(embed_dim)
# # 두 번째 위치에 있는 토큰에 로터리 위치 임베딩을 적용합니다.
# rotary_position_embedding(inputs, 1)

(1, 2, 4096)
tf.Tensor(
[[[ 1.          1.          1.         ...  1.          1.
    1.        ]
  [-0.30116868 -0.2949654  -0.28878427 ...  1.0001013   1.0001009
    1.0001005 ]]], shape=(1, 2, 4096), dtype=float32)


In [6]:
rotary_embedding = RotaryEmbedding(embed_dim)

x = tf.ones((1, 2, embed_dim))
rotary_embedding(x)

# rotary_embedding = keras_nlp.layers.RotaryEmbedding()
# rotary_embedding(keras.ops.ones((1, 2, embed_dim)))

<tf.Tensor: shape=(1, 2, 4096), dtype=float32, numpy=
array([[[ 1.        ,  1.        ,  1.        , ...,  1.        ,
          1.        ,  1.        ],
        [-0.30116868, -0.2949654 , -0.28878427, ...,  1.0001013 ,
          1.0001009 ,  1.0001005 ]]], dtype=float32)>

### RMS 정규화

In [7]:
import numpy as np
import tensorflow as tf

def rms_norm(x):
    x = tf.cast(x, tf.float32)
    scale = 1.0
    epsilon = 1e-6
    var = tf.reduce_mean(tf.pow(x, 2), axis=-1, keepdims=True)
    return scale * x / tf.sqrt(var + epsilon)

x = np.array([1, 2, 3])
print(rms_norm(x).numpy())

# import numpy as np

# def rms_norm(x):
#     scale = 1.0     # 실제로는 훈련되는 가중치입니다.
#     epsilon = 1e-6
#     var = keras.ops.mean(keras.ops.power(x, 2), axis=-1, keepdims=True)
#     return scale * x / keras.ops.sqrt(var + epsilon)

# x = np.array([1, 2, 3])
# rms_norm(x)

[0.46291 0.92582 1.38873]


In [8]:
import tensorflow as tf
from tensorflow.keras import layers

class LlamaRMSNorm(layers.Layer):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.hidden_size = hidden_size
        self.eps = eps

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="weight",
            shape=(self.hidden_size,),
            initializer="ones",
            trainable=True,
        )

    def call(self, x):
        x = tf.cast(x, tf.float32)
        variance = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)
        x = x * tf.math.rsqrt(variance + self.eps)
        return x * self.weight

llama_norm = LlamaRMSNorm(hidden_size=x.shape[-1])
y = llama_norm(x)
print(y.shape)

# from keras_nlp.src.models.llama.llama_layernorm import LlamaLayerNorm

# llama_norm = LlamaLayerNorm()
# llama_norm(x)

(3,)


### SwiGLU 활성화 함수

In [9]:
# 피드 포워드 네트워크의 입력 크기가 (10, 4096)이고,
# 유닛 개수는 11,008개, 임베딩 차원은 4,096이라고 가정합니다.
x = tf.ones((10, 4096))
x1 = layers.Dense(11008, activation='silu', use_bias=False)(x)
x2 = layers.Dense(11008, use_bias=False)(x)
x = x1 * x2
x = layers.Dense(4096, use_bias=False)(x)
x

<tf.Tensor: shape=(10, 4096), dtype=float32, numpy=
array([[-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822],
       [-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822],
       [-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822],
       ...,
       [-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822],
       [-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822],
       [-0.1181376 ,  0.21388917,  0.06665327, ..., -0.0182859 ,
         0.13931775, -0.21030822]], dtype=float32)>

## KerasNLP로 Llama-2 모델 만들기

In [10]:
import tensorflow as tf
from tensorflow.keras import layers

def make_attention_mask(padding_mask):
    # padding_mask: (batch, seq_len), 1=valid, 0=padding
    padding_mask = tf.cast(padding_mask, tf.float32)
    return padding_mask[:, tf.newaxis, tf.newaxis, :]  # (batch, 1, 1, seq_len)

class AttentionMask(layers.Layer):
    def call(self, padding_mask):
        return make_attention_mask(padding_mask)

class LlamaRMSNorm(layers.Layer):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.hidden_size = hidden_size
        self.eps = eps

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="weight",
            shape=(self.hidden_size,),
            initializer="ones",
            trainable=True,
        )

    def call(self, x):
        x = tf.cast(x, tf.float32)
        variance = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)
        x = x * tf.math.rsqrt(variance + self.eps)
        return x * self.weight

def llama_decoder(x, padding_mask, num_query_heads, num_key_value_heads,
                  interm_dim, hidden_dim):
    # attention mask
    attention_mask = AttentionMask()(padding_mask)      # (batch,1,1,seq_len)
    attention_mask = tf.squeeze(attention_mask, axis=1) # (batch,1,seq_len)

    # attention block
    residual = x
    x = LlamaRMSNorm(hidden_dim)(x)

    llama_attention = layers.MultiHeadAttention(
        num_heads=num_query_heads,
        key_dim=hidden_dim // num_query_heads,
        use_bias=False,
        dropout=0.0,
    )
    x = llama_attention(x, x, attention_mask=attention_mask)
    x = x + residual

    # feed-forward block (SwiGLU 스타일)
    residual = x
    x = LlamaRMSNorm(hidden_dim)(x)

    x1 = layers.Dense(interm_dim, activation=tf.nn.silu, use_bias=False)(x)
    x2 = layers.Dense(interm_dim, use_bias=False)(x)
    x = x1 * x2
    x = layers.Dense(hidden_dim, use_bias=False)(x)

    x = x + residual
    return x

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# LLaMA 2
vocab_size = 32000
num_layers = 32
num_query_heads = 32
num_key_value_heads = 32
interm_dim = 11008
hidden_dim = 4096

def make_attention_mask(padding_mask):
    padding_mask = tf.cast(padding_mask, tf.float32)
    return padding_mask[:, tf.newaxis, tf.newaxis, :]

class AttentionMask(layers.Layer):
    def call(self, padding_mask):
        return make_attention_mask(padding_mask)

class LlamaRMSNorm(layers.Layer):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.hidden_size = hidden_size
        self.eps = eps

    def build(self, input_shape):
        self.weight = self.add_weight(
            name="weight",
            shape=(self.hidden_size,),
            initializer="ones",
            trainable=True,
        )

    def call(self, x):
        x = tf.cast(x, tf.float32)
        variance = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True)
        x = x * tf.math.rsqrt(variance + self.eps)
        return x * self.weight

def llama_decoder(x, padding_mask, num_query_heads, num_key_value_heads,
                  interm_dim, hidden_dim):
    attention_mask = AttentionMask()(padding_mask)
    attention_mask = tf.squeeze(attention_mask, axis=1)

    residual = x
    x = LlamaRMSNorm(hidden_dim)(x)

    x = layers.MultiHeadAttention(
        num_heads=num_query_heads,
        key_dim=hidden_dim // num_query_heads,
        use_bias=False,
        dropout=0.0,
    )(x, x, attention_mask=attention_mask)

    x = x + residual

    residual = x
    x = LlamaRMSNorm(hidden_dim)(x)

    x1 = layers.Dense(interm_dim, activation=tf.nn.silu, use_bias=False)(x)
    x2 = layers.Dense(interm_dim, use_bias=False)(x)
    x = x1 * x2
    x = layers.Dense(hidden_dim, use_bias=False)(x)

    x = x + residual
    return x

token_ids = keras.Input(shape=(None,), dtype="int32", name="token_ids")
padding_mask = keras.Input(shape=(None,), dtype="float32", name="padding_mask")

# ReversibleEmbedding 대신 일반 Embedding
token_embedding_layer = layers.Embedding(
    input_dim=vocab_size,
    output_dim=hidden_dim,
    name="token_embedding"
)

x = token_embedding_layer(token_ids)

for _ in range(num_layers):
    x = llama_decoder(
        x,
        padding_mask,
        num_query_heads,
        num_key_value_heads,
        interm_dim,
        hidden_dim,
    )

x = LlamaRMSNorm(hidden_dim)(x)

# reverse=True 대신 출력 projection
outputs = layers.Dense(vocab_size, use_bias=False, name="lm_head")(x)

model = keras.Model(
    inputs=(token_ids, padding_mask),
    outputs=outputs,
)

model.summary(line_length=120)

Model: "model"
________________________________________________________________________________________________________________________
 Layer (type)                          Output Shape               Param #       Connected to                            
 token_ids (InputLayer)                [(None, None)]             0             []                                      
                                                                                                                        
 padding_mask (InputLayer)             [(None, None)]             0             []                                      
                                                                                                                        
 token_embedding (Embedding)           (None, None, 4096)         131072000     ['token_ids[0][0]']                     
                                                                                                                        
 attention_mask (

c:\Users\an9383\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\utils\layer_utils.py:124: RuntimeWarning: overflow encountered in long_scalars
  return int(sum(np.prod(p) for p in standardized_weight_shapes))
